In [51]:
import pandas as pd
import ccxt
import plotly
print("ccxt:", ccxt.__version__)
print("pandas:", pd.__version__)
print("plotly:", plotly.__version__)


ccxt: 4.5.88
pandas: 3.0.6
plotly: 7.1.0


In [52]:
exchange = ccxt.binance()

print("Rate limiting enabled:", exchange.enableRateLimit)
print("Minimum delay between requests (ms):", exchange.rateLimit)

Rate limiting enabled: True
Minimum delay between requests (ms): 50


In [53]:
candles = exchange.fetch_ohlcv("BTC/USDT", timeframe="1d", limit=5)

print(type(candles), len(candles))
for candle in candles:
    print(candle)

<class 'list'> 5
[1791849600000, 81850.01, 84168.0, 81033.66, 83169.0, 14784.0627]
[1791936000000, 83169.01, 84146.72, 81444.42, 81682.01, 18307.57283]
[1792022400000, 81682.02, 82634.3, 79800.0, 82245.93, 22563.14627]
[1792108800000, 82245.93, 82940.81, 80809.25, 81615.64, 14753.9576]
[1792195200000, 81615.91, 81770.01, 81416.0, 81696.26, 2327.15006]


In [54]:
for i in range(len(candles)-1) :
    date=pd.to_datetime(candles[i][0], unit="ms", utc=True)
    print(date, "close:", candles[i][4], "next open:", candles[i+1][1])
    print("------------")
print(pd.to_datetime(candles[-1][0], unit="ms", utc=True), "close:", candles[-1][4])

2026-10-13 00:00:00+00:00 close: 83169.0 next open: 83169.01
------------
2026-10-14 00:00:00+00:00 close: 81682.01 next open: 81682.02
------------
2026-10-15 00:00:00+00:00 close: 82245.93 next open: 82245.93
------------
2026-10-16 00:00:00+00:00 close: 81615.64 next open: 81615.91
------------
2026-10-17 00:00:00+00:00 close: 81696.26


In [55]:
all_candles = []
since = exchange.parse8601("2017-01-01T00:00:00Z")

while True:
    batch = exchange.fetch_ohlcv("BTC/USDT", timeframe="1d", since=since, limit=1000)
    if len(batch) == 0:
        break
    all_candles.extend(batch)

    first_date = pd.to_datetime(batch[0][0], unit="ms", utc=True)
    last_date = pd.to_datetime(batch[-1][0], unit="ms", utc=True)
    print(f"Fetched {len(batch)} candles: {first_date.date()} -> {last_date.date()}")

    if len(batch) < 1000:
        break
    since = batch[-1][0] + 1

print("Total candles:", len(all_candles))

Fetched 1000 candles: 2017-08-17 -> 2020-05-12
Fetched 1000 candles: 2020-05-13 -> 2023-02-06
Fetched 1000 candles: 2023-02-07 -> 2025-11-02
Fetched 349 candles: 2025-11-03 -> 2026-10-17
Total candles: 3349


In [56]:
## Step 4 — Build the DataFrame
columns=["timestamp", "open", "high", "low", "close", "volume"]
df=pd.DataFrame(all_candles, columns=columns)

df["date"]=pd.to_datetime(df["timestamp"], unit="ms", utc=True)
df=df.set_index("date")

In [57]:
df.tail()

,timestamp,open,high,low,close,volume
date,,,,,,
2026-10-13 00:00:00+00:00,1791849600000,81850.01,84168.00,81033.66,83169.00,14784.06270
2026-10-14 00:00:00+00:00,1791936000000,83169.01,84146.72,81444.42,81682.01,18307.57283
2026-10-15 00:00:00+00:00,1792022400000,81682.02,82634.30,79800.00,82245.93,22563.14627
2026-10-16 00:00:00+00:00,1792108800000,82245.93,82940.81,80809.25,81615.64,14753.95760
2026-10-17 00:00:00+00:00,1792195200000,81615.91,81770.01,81416.00,81696.26,2327.16213


In [58]:
DAY_MS = 86_400_000
price_cols = ["open", "high", "low", "close"]

# 1. Duplicate timestamps
print("Duplicate dates:", int(df.index.duplicated().sum()))

# 2. Chronological order
print("Sorted oldest -> newest:", df.index.is_monotonic_increasing)

# 3. Gaps
diffs = df["timestamp"].diff()
gaps = diffs[diffs > DAY_MS]
missing_days = int((gaps / DAY_MS - 1).sum())
print("Gaps:", len(gaps), "| missing days:", missing_days)

# 4. Missing values
for col in df.columns :
    na_values=df[col].isna()
    print("Missing values of ",col," :", na_values.sum())

# 5. Invalid values
print("Non-positive prices:", int((df[price_cols] <= 0).sum().sum()))
print("Negative volume:", int((df["volume"] < 0).sum()))
print("Zero-volume days:", int((df["volume"] == 0).sum()))

# 6. Candle consistency
bad_high = df["high"] < df[["open", "close"]].max(axis=1)
bad_low = df["low"] > df[["open", "close"]].min(axis=1)
print("High below open/close:", int(bad_high.sum()))
print("Low above open/close:", int(bad_low.sum()))

Duplicate dates: 0
Sorted oldest -> newest: True
Gaps: 0 | missing days: 0
Missing values of  timestamp  : 0
Missing values of  open  : 0
Missing values of  high  : 0
Missing values of  low  : 0
Missing values of  close  : 0
Missing values of  volume  : 0
Non-positive prices: 0
Negative volume: 0
Zero-volume days: 0
High below open/close: 0
Low above open/close: 0


In [59]:
now_ms = exchange.milliseconds()
is_complete = df["timestamp"] + DAY_MS <= now_ms

print("Incomplete candles:", int((~is_complete).sum()))
print(df[~is_complete])

df = df[is_complete]
print("Rows after removal:", len(df), "| last date:", df.index[-1].date())

Incomplete candles: 1
                               timestamp      open      high      low  \
date                                                                    
2026-10-17 00:00:00+00:00  1792195200000  81615.91  81770.01  81416.0   

                              close      volume  
date                                             
2026-10-17 00:00:00+00:00  81696.26  2327.16213  
Rows after removal: 3348 | last date: 2026-10-16


In [60]:
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_path = RAW_DIR / "btc_usdt_1d_raw.csv"
processed_path = PROCESSED_DIR / "btc_usdt_1d.csv"

raw_df = pd.DataFrame(all_candles, columns=["timestamp", "open", "high", "low", "close", "volume"])
raw_df.to_csv(raw_path, index=False)
df.to_csv(processed_path)

print("Raw:      ", raw_path.resolve(), "|", len(raw_df), "rows")
print("Processed:", processed_path.resolve(), "|", len(df), "rows")

Raw:       C:\projects\CryptoSense AI\data\raw\btc_usdt_1d_raw.csv | 3349 rows
Processed: C:\projects\CryptoSense AI\data\processed\btc_usdt_1d.csv | 3348 rows


In [61]:
reloaded = pd.read_csv(processed_path)
reloaded["date"] = pd.to_datetime(reloaded["timestamp"], unit="ms", utc=True)
reloaded = reloaded.set_index("date")

pd.testing.assert_frame_equal(df, reloaded)
print("Round trip OK:", reloaded.shape, reloaded.index.dtype)

Round trip OK: (3348, 6) datetime64[ms, UTC]


In [62]:
## Step 6 — Visualize
import plotly.express as px

fig = px.line(df, y="close", title="BTC/USDT daily close")
fig.update_yaxes(type="log")
fig.show()

In [63]:
fig = px.bar(x=df.index, y=df["volume"] * df["close"],
             title="BTC/USDT daily volume (USDT, approx.)",
             labels={"x": "date", "y": "USDT"})
fig.show()